# 01 · Preparar datos (entorno CPU)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tomas-pucv/Camanchaca-Predict/blob/main/notebooks/01_prepare_data.ipynb)

**Propósito.** Descargar y preprocesar las fuentes (RESIDE-OTS, Foggy Cityscapes, CS-fog con
profundidad estéreo, autos de Cityscapes, HazeRD, O-HAZE y fotos propias de La Pólvora), crear los
splits sin fuga y guardar las imágenes preprocesadas como `.tar` en Drive.

**Tiempo esperado.** 2–4 h, dominado por descargas (RESIDE ~12 GB: 3–10 min; Cityscapes ~45 GB:
30–90 min) y preprocesamiento (~30–60 min). Re-ejecutable: los pasos reutilizan lo ya hecho.
**Entorno.** **CPU** (no gastar cuota de GPU). Revisar el disco libre con `df -h` (~70–100 GB).

**Salidas.**
- `data/manifests/*.csv` y `data/splits/v1/{train,val,test,ext}.csv` + `SPLIT_INFO.json` (se versionan).
- `reports/tables/data_counts.{md,csv}` (n por split × fuente × banda).
- `MyDrive/camanchaca/data/<fuente>.tar` (imágenes preprocesadas; los demás notebooks las desempaquetan).

## 0. Accesos (pedir HOY: algunos tardan días)
| Fuente | Acceso | Notas |
|---|---|---|
| RESIDE-OTS | token de Kaggle (`kaggle.json`) | niebla sintética sobre fotos reales: etiqueta = *visibilidad equivalente* |
| Cityscapes + Foggy Cityscapes | registro en cityscapes-dataset.com con correo institucional | aprobación manual (días); prohibido redistribuir imágenes |
| HazeRD | formulario Google del sitio de G. Sharma (U. Rochester) o IEEE DataPort | solo evaluación; confirmar ε en `demo_simu_haze.m` |
| O-HAZE | descarga pública (NTIRE 2018) | niebla real de máquina; solo evaluación |
| La Pólvora | fotos propias (`MyDrive/camanchaca/raw/polvora/`) | solo evaluación; `labels.csv` opcional con V medida |

Credenciales en `MyDrive/camanchaca/` (`kaggle.json`, `cityscapes_credentials.json`), nunca en git.

## Configuración común
Misma celda en todos los notebooks (detalles en `00_setup_colab`). En **Colab**: monta Drive,
fija `CAMANCHACA_DATA=/content/data` y `CAMANCHACA_RUNS=/content/drive/MyDrive/camanchaca/runs`,
clona/actualiza el repo en la referencia `REF`, instala el paquete (`pip install -e ".[demo,data]"`,
**sin** reinstalar torch) y desempaqueta los `.tar` de datos de Drive (salvo `NEEDS_DATA = False`). **Fuera de Colab** (`SMOKE=True`)
usa datos sintéticos y pesos aleatorios generados por `camanchaca smoke`: sirve para verificar que
todo corre, **sus números no son resultados**.

In [ ]:
# Configuración común (idéntica en los notebooks 00-06)
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SMOKE = not IN_COLAB   # True: datos sintéticos + pesos aleatorios (solo verifica que todo corre)
REF = "main"           # rama/tag/commit del repo; para la entrega final usar "v1.0-final"
REPO_URL = "https://github.com/Tomas-pucv/Camanchaca-Predict.git"
DRIVE_DIR = Path("/content/drive/MyDrive/camanchaca")   # tars de datos, corridas, pesos y credenciales

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    os.environ["CAMANCHACA_DATA"] = "/content/data"
    os.environ["CAMANCHACA_RUNS"] = str(DRIVE_DIR / "runs")
    REPO = Path("/content/Camanchaca-Predict")
    if not REPO.exists():
        !git clone -q {REPO_URL} {REPO}
    !git -C {REPO} fetch -q --all --tags && git -C {REPO} checkout -q {REF} && (git -C {REPO} pull -q --ff-only 2>/dev/null || true)
    %cd {REPO}
    !pip install -q -e ".[demo,data]"   # NO reinstala torch/torchvision: se usan los de Colab
    if not SMOKE and globals().get("NEEDS_DATA", True):
        for tar in sorted((DRIVE_DIR / "data").glob("*.tar")):   # imágenes preprocesadas (notebook 01)
            !camanchaca unpack --tar {tar}
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    %cd {REPO}
    if shutil.which("camanchaca") is None:
        print("AVISO: falta el comando `camanchaca`: ejecutar  pip install -e '.[dev,demo]'")

WORK = Path(os.environ.get("CAMANCHACA_SMOKE_DIR", str(REPO / "local_workspace" / "nb_smoke")))
if SMOKE:
    os.environ.update(CAMANCHACA_DATA=str(WORK / "data"), CAMANCHACA_RUNS=str(WORK / "runs"),
                      CAMANCHACA_REPORTS=str(WORK / "reports"), HF_HUB_OFFLINE="1")
    if not (WORK / "smoke.yaml").exists():   # ~4-6 min en CPU, una sola vez
        !camanchaca smoke --workdir {WORK}

sys.path.insert(0, str(REPO / "src"))   # importable sin reiniciar el kernel tras `pip install -e`
from camanchaca.config import load_yaml
from camanchaca.paths import data_root, reports_dir, runs_root

EXP = str(WORK / "smoke.yaml") if SMOKE else "configs/experiment/main.yaml"
EXP_NAME = load_yaml(EXP)["name"]
print(f"Colab={IN_COLAB} | SMOKE={SMOKE} | EXP={EXP}")
print(f"datos={data_root()} | corridas={runs_root()} | reportes={reports_dir()}")

## 1. RESIDE-OTS (Kaggle, ~12 GB)
**VERIFICAR el slug** en kaggle.com antes de descargar: debe ser el OTS oficial con nombres
`<id>_<A>_<beta>.jpg`. `--check-only` informa el % de nombres válidos, betas/A y escenas, y
**aborta** si parece un reempaque (p. ej. RESIDE-6K): en ese caso no continuar y buscar otro mirror.

In [ ]:
KAGGLE_SLUG = "brunobelloni/outdoor-training-set-ots-reside"   # VERIFICAR antes de descargar
RAW = Path("/content/raw")
if not SMOKE:
    !mkdir -p ~/.kaggle {RAW}/reside && cp {DRIVE_DIR}/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download {KAGGLE_SLUG} -p {RAW}/reside
    !ls -lh {RAW}/reside && df -h /content

In [ ]:
RESIDE_ZIP = next((RAW / "reside").glob("*.zip"), RAW / "reside")
if not SMOKE:
    !camanchaca prep reside --raw {RESIDE_ZIP} --check-only

In [ ]:
if not SMOKE:
    !camanchaca prep reside --raw {RESIDE_ZIP} --one-a-per-beta --workers 2
    !rm -f {RESIDE_ZIP}   # libera ~12 GB

## 2. Cityscapes (≈45 GB en total)
`cityscapesscripts` lee las credenciales de `~/.local/share/cityscapesscripts/credentials.json`.
Tamaños aproximados (**verificar con `csDownload -l`**):

| Paquete | Tamaño | Uso |
|---|---|---|
| `leftImg8bit_trainvaltest_foggy.zip` | ~30 GB | Foggy Cityscapes y autos con niebla (se lee **sin extraer**) |
| `leftImg8bit_trainvaltest.zip` | ~11 GB | imágenes limpias (CS-fog, autos) |
| `disparity_trainvaltest.zip` | ~3,5 GB | profundidad estéreo (CS-fog, distancia de autos) |
| `gtFine_trainvaltest.zip` | ~0,25 GB | clases (cielo/calzada/vehículo) e instancias de autos |
| `camera_trainvaltest.zip` | ~2 MB | calibración (baseline, focal) |

El disco de Colab no alcanza para todo a la vez: descargar en este orden, extraer solo
`train`/`val` (el *test* oficial no trae etiquetas) y **borrar cada zip** al terminar.

In [ ]:
CS = Path("/content/cs")
if not SMOKE:
    !mkdir -p ~/.local/share/cityscapesscripts {CS}
    !cp {DRIVE_DIR}/cityscapes_credentials.json ~/.local/share/cityscapesscripts/credentials.json
    !csDownload -l
    !df -h /content

### 2a. Imágenes limpias + geometría → CS-fog
Niebla sintética con profundidad estéreo real y V muestreada log-uniforme en 15–1000 m.

In [ ]:
CS_PACKAGES = ["gtFine_trainvaltest.zip", "camera_trainvaltest.zip",
               "disparity_trainvaltest.zip", "leftImg8bit_trainvaltest.zip"]
if not SMOKE:
    for pkg in CS_PACKAGES:
        !csDownload -d {CS} {pkg}
        !cd {CS} && unzip -q -o {pkg} -x "*/test/*" && rm -f {pkg}
    !du -sh {CS}/* && df -h /content
    !camanchaca prep cs-fog --cs-root {CS} --workers 2

### 2b. Foggy Cityscapes (zip de ~30 GB, sin extraer) y autos
`prep foggy-cityscapes --zip` lee los PNG directamente del zip (β = 0,005/0,01/0,02 → MOR
600/300/150 m). `prep cars` usa el split oficial *val* (ciudades de test del proyecto) para el
análisis de detección de autos vs niebla (notebook 05). Opcional: `--refined-list` marca el
subconjunto "refined" de Foggy Cityscapes.

In [ ]:
FOGGY_ZIP = CS / "leftImg8bit_trainvaltest_foggy.zip"
if not SMOKE:
    !csDownload -d {CS} {FOGGY_ZIP.name}
    !camanchaca prep foggy-cityscapes --zip {FOGGY_ZIP} --workers 2
    !camanchaca prep cars --cs-root {CS} --foggy-zip {FOGGY_ZIP} --split val --workers 2
    !rm -f {FOGGY_ZIP}   # libera ~30 GB

## 3. Pruebas externas (solo evaluación): HazeRD, O-HAZE y La Pólvora
- **HazeRD**: subir el zip recibido por el formulario a `MyDrive/camanchaca/raw/HazeRD.zip`.
  `--eps 0.02` es la convención supuesta de su script: **confirmar** en `demo_simu_haze.m`.
- **O-HAZE**: **VERIFICAR** la URL en la página de NTIRE 2018.
- **La Pólvora**: fotos propias en `MyDrive/camanchaca/raw/polvora/` (+ `labels.csv` opcional).

In [ ]:
HAZERD_ZIP = DRIVE_DIR / "raw" / "HazeRD.zip"
OHAZE_URL = "https://data.vision.ee.ethz.ch/cvl/ntire18/o-haze/O-HAZE.zip"   # VERIFICAR
POLVORA_DIR = DRIVE_DIR / "raw" / "polvora"
LABELS = POLVORA_DIR / "labels.csv"
LABEL_ARGS = f"--labels {LABELS}" if LABELS.exists() else ""
if not SMOKE:
    !mkdir -p {RAW}/hazerd && unzip -q -o {HAZERD_ZIP} -d {RAW}/hazerd
    !camanchaca prep hazerd --raw {RAW}/hazerd --eps 0.02
    !wget -q -O {RAW}/O-HAZE.zip {OHAZE_URL}
    !camanchaca prep ohaze --raw {RAW}/O-HAZE.zip
    if POLVORA_DIR.exists():
        !camanchaca prep folder --raw {POLVORA_DIR} --name polvora {LABEL_ARGS}

## 4. Splits sin fuga
Cityscapes (Foggy y CS-fog) se separa por **ciudad** (test = frankfurt, lindau, munster);
RESIDE por **escena**; HazeRD, O-HAZE y La Pólvora van a `ext` (solo evaluación).

In [ ]:
if not SMOKE:
    !camanchaca split

In [ ]:
import pandas as pd
from IPython.display import Markdown, display
SPLITS = Path(load_yaml(EXP)["data"]["splits_dir"])
SPLITS = SPLITS if SPLITS.is_absolute() else REPO / SPLITS
COUNTS = pd.concat({s: pd.read_csv(SPLITS / f"{s}.csv") for s in ["train", "val", "test", "ext"]})
display(COUNTS.groupby(level=0)["source"].value_counts().unstack(fill_value=0))
DATA_COUNTS = reports_dir() / "tables" / "data_counts.md"
if DATA_COUNTS.exists(): display(Markdown(DATA_COUNTS.read_text()))

## 5. Empaquetar en Drive
Un `.tar` **sin compresión** por fuente (más `aux/`, con distancias y clases de CS-fog). Los
notebooks 02–06 los desempaquetan a `/content/data` al inicio (~1 min/GB): leer miles de archivos
pequeños directamente desde Drive es demasiado lento.

In [ ]:
TARS = WORK / "tars" if SMOKE else DRIVE_DIR / "data"
SOURCES = sorted(p for p in (data_root() / "images").iterdir() if p.is_dir())
SOURCES += [p for p in [data_root() / "aux"] if p.is_dir()]
for src in SOURCES:
    !camanchaca pack --src {src} --tar {TARS}/{src.name}.tar
!ls -lh {TARS}

In [ ]:
if SMOKE:   # verificación de ida y vuelta (en Colab lo hace la configuración común)
    FIRST_TAR = sorted(TARS.glob("*.tar"))[0]
    !camanchaca unpack --tar {FIRST_TAR} --dest {WORK}/unpack_check

## 6. Versionar manifiestos y splits
Los CSV son pequeños y **sí** se versionan (rutas relativas a `CAMANCHACA_DATA`; nunca imágenes).
En Colab esta celda los deja en `MyDrive/camanchaca/manifests_splits.zip` (incluye
`cars_gt.csv`/`cars_images.csv` de `prep cars`, que usa el notebook 05).

In [ ]:
if not SMOKE:
    !zip -qr {DRIVE_DIR}/manifests_splits.zip data/manifests data/splits reports/tables/data_counts.md reports/tables/data_counts.csv

## 7. Commit (en el PC, una vez)
Descargar `manifests_splits.zip` de Drive, descomprimirlo en la raíz del repo y versionar los
manifiestos y los splits (los notebooks 02–06 los leen del repo clonado en `REF`):

```bash
unzip -o manifests_splits.zip
git add data/manifests/*.csv data/splits/v1/* reports/tables/data_counts.*
git commit -m "Datos: manifiestos y splits v1"
git push
```
Registrar en el README el sha256 de `test.csv` y `ext.csv` (están en `SPLIT_INFO.json`).